![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [1]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  1


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [2]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]], shape=(442, 10)),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142.

We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [3]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [5]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [6]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [7]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
# Fix the seed before creating the original network.
torch.manual_seed(42)
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [8]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 34212.0234
Epoch [10/100], Loss: 30378.7988
Epoch [10/100], Loss: 23441.7578
Epoch [10/100], Loss: 33402.9688
Epoch [10/100], Loss: 31066.1523
Epoch [10/100], Loss: 28054.377
Epoch [10/100], Loss: 26669.0996
Epoch [10/100], Loss: 32195.5781
Epoch [20/100], Loss: 28172.627
Epoch [20/100], Loss: 30717.1289
Epoch [20/100], Loss: 31766.1152
Epoch [20/100], Loss: 24998.8203
Epoch [20/100], Loss: 27522.6348
Epoch [20/100], Loss: 39604.1055
Epoch [20/100], Loss: 25163.4336
Epoch [20/100], Loss: 12928.0732
Epoch [30/100], Loss: 28245.9473
Epoch [30/100], Loss: 32426.3145
Epoch [30/100], Loss: 27819.2168
Epoch [30/100], Loss: 27402.5996
Epoch [30/100], Loss: 22780.209
Epoch [30/100], Loss: 38466.8203
Epoch [30/100], Loss: 30583.1523
Epoch [30/100], Loss: 5867.3135
Epoch [40/100], Loss: 33800.7734
Epoch [40/100], Loss: 29845.0684
Epoch [40/100], Loss: 31020.3984
Epoch [40/100], Loss: 35297.5234
Epoch [40/100], Loss: 27741.1602
Epoch [40/100], Loss: 27011.8086
Epoch [40/100]

We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:

In [9]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 23429.6943359375


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [10]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,10.406027,219.0
1,9.857447,70.0
2,10.189686,202.0
3,12.422951,230.0
4,9.824396,111.0
...,...,...
84,8.955412,153.0
85,8.328184,98.0
86,7.637214,37.0
87,7.868061,63.0


Side-by-side, they don't look great. Can you improve them?

<br><br>

## EXERCISE #1
Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?


In [11]:
mse_100 = avg_mse

model.train()

for epoch in range(100, 1000):
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimiser.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimiser.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/1000 completed")

Epoch 200/1000 completed
Epoch 300/1000 completed
Epoch 400/1000 completed
Epoch 500/1000 completed
Epoch 600/1000 completed
Epoch 700/1000 completed
Epoch 800/1000 completed
Epoch 900/1000 completed
Epoch 1000/1000 completed


In [12]:
model.eval()
mse_values = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        mse_values.append(criterion(model(inputs), targets).item())

mse_1000 = np.mean(mse_values)

print("MSE after 100 epochs:", round(mse_100, 2))
print("MSE after 1000 epochs:", round(mse_1000, 2))

MSE after 100 epochs: 23429.69
MSE after 1000 epochs: 2953.88


## Exercise #1: Increasing Training Epochs

I trained the model for 100 epochs and then continued training for another 900, keeping the same network architecture and data split.

The following cell calculates the comparison from the latest evaluation results.


In [13]:
print(f"Test MSE after 100 epochs: {mse_100:,.2f}")
print(f"Test MSE after 1000 epochs: {mse_1000:,.2f}")

if mse_1000 < mse_100:
    reduction = (mse_100 - mse_1000) / mse_100 * 100
    print(
        f"Increasing training from 100 to 1000 epochs reduced the test MSE "
        f"by {reduction:.1f}% in this run."
    )
elif mse_1000 > mse_100:
    print("Increasing training to 1000 epochs did not improve the test MSE in this run.")
else:
    print("The test MSE was unchanged after increasing training to 1000 epochs.")


Test MSE after 100 epochs: 23,429.69
Test MSE after 1000 epochs: 2,953.88
Increasing training from 100 to 1000 epochs reduced the test MSE by 87.4% in this run.



## EXERCISE #2 (optional)
Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?

In [14]:
torch.manual_seed(42)

# Create a wider network
wide_model = nn.Sequential(
    nn.Linear(10, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
).to(device)

wide_optimiser = optim.Adam(wide_model.parameters(), lr=0.001)

# Train the new network from scratch for 1000 epochs
wide_model.train()

for epoch in range(1000):
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        wide_optimiser.zero_grad()
        loss = criterion(wide_model(inputs), targets)
        loss.backward()
        wide_optimiser.step()

    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch + 1}/1000 completed")

# Evaluate using the teacher's batch-average MSE method
wide_model.eval()
wide_mse_values = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        wide_mse_values.append(
            criterion(wide_model(inputs), targets).item()
        )

wide_mse = np.mean(wide_mse_values)

print("Original network MSE:", round(mse_1000, 2))
print("Wider network MSE:", round(wide_mse, 2))

Epoch 200/1000 completed
Epoch 400/1000 completed
Epoch 600/1000 completed
Epoch 800/1000 completed
Epoch 1000/1000 completed
Original network MSE: 2953.88
Wider network MSE: 2868.71


### Results and Discussion

I increased the hidden layer sizes from 5 and 5 neurons to 16 and 8 neurons, keeping the same data split, learning rate and total training epochs.

The following cell compares the latest test MSE values. A single comparison is not sufficient to establish an optimal architecture.


In [15]:
print(f"Original network test MSE: {mse_1000:,.2f}")
print(f"Wider network test MSE: {wide_mse:,.2f}")

if wide_mse < mse_1000:
    print("The wider network had a lower test MSE in this run.")
elif wide_mse > mse_1000:
    print("The original network had a lower test MSE in this run.")
else:
    print("Both networks had the same test MSE in this run.")

print(
    "This single comparison does not establish an optimal architecture. "
    "Random initialisation and the training order may also affect the result."
)


Original network test MSE: 2,953.88
Wider network test MSE: 2,868.71
The wider network had a lower test MSE in this run.
This single comparison does not establish an optimal architecture. Random initialisation and the training order may also affect the result.
